# Quant Rush · Workshop 1: Picks & Shovels

From zero to a submitted baseline. This notebook covers:

1. **Just enough Python:** values, lists, functions.
2. **pandas:** loading the data, looking at it, picking rows and columns, grouping by era.
3. **A first prediction:** a one-line rule, checked locally with the competition metric.
4. **A submission file:** written and checked before you upload it.

No machine learning yet. Run each cell with **Shift + Enter**, top to bottom.

**How to run it.** Either:

- **On Kaggle:** create a new notebook, choose **File → Import Notebook** and upload this file, then use **+ Add Input** to attach the Quant Rush competition data.
- **On your computer:** download the competition files from the competition's Data tab, and put `train.parquet` and `test.parquet` in the same folder as this notebook (or a folder inside it). You need Python with pandas and pyarrow.

## 1. Just enough Python

A notebook runs Python one cell at a time. Values get names, lists hold several values, and functions are reusable steps.

In [ ]:
hours_of_history = 12
channels = ["ret", "vol", "range", "taker"]
print(hours_of_history, "hours of", len(channels), "channels:", channels)

In [ ]:
def column_names(channel):
    """All 12 column names of one channel, from the latest hour (00) to the oldest (11)."""
    return [f"{channel}_{k:02d}" for k in range(hours_of_history)]


column_names("ret")

## 2. Load the data with pandas

**pandas** is the Python library for tables. A table is a **DataFrame**; one column of it is a **Series**. The competition files come as CSV and as parquet; parquet loads much faster.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Find the folder with the competition files: on Kaggle it is under /kaggle/input.
DATA = next(p.parent for p in [*Path("/kaggle/input").rglob("train.parquet"), *Path(".").rglob("train.parquet")])

train = pd.read_parquet(DATA / "train.parquet")
test = pd.read_parquet(DATA / "test.parquet")
print("train:", train.shape, "  test:", test.shape)

In [ ]:
train.head()

In [ ]:
print(list(train.columns[:6]), "...", list(train.columns[-3:]))

Each row is one **asset** in one **era** (a moment in time). The columns:

| Columns | What they are |
|---|---|
| `id` | Row id. Your submission uses it. |
| `era` | The moment the row belongs to. Train eras are numbered in time order. |
| `ret_00` … `ret_11` | The asset's return in each of the last 12 hours, compared with the other assets in the era. |
| `vol_`, `range_`, `taker_` | Trading volume, high–low range, and the share of aggressive buyers, each hour, compared the same way. |
| `mkt_ret_`, `mkt_disp_` | The whole market each hour. The same on every row of an era. |
| `target` (train only) | How the asset did over the next 4 hours, as a rank within the era from −0.5 (worst) to +0.5 (best). |

The test set has the same columns except `target`: that is what you predict.

## 3. Picking columns and rows

Square brackets pick columns. A condition inside square brackets picks rows.

In [ ]:
train["ret_00"].head()  # one column: a Series

In [ ]:
train[["id", "era", "ret_00", "target"]].head()  # a list of columns: a DataFrame

In [ ]:
first_era = train[train["era"] == 0]  # only the rows where the condition is True
print("rows in era 0:", len(first_era))
first_era[["ret_00", "target"]].describe().round(3)

## 4. Eras: grouping

`groupby` splits the table by a column and summarizes each group. Here: how many assets each era has, and the average target in each era.

In [ ]:
per_era = train.groupby("era").size()
print("train eras:", len(per_era), "  assets per era:", per_era.min(), "to", per_era.max())
print("test eras:", test["era"].nunique())

In [ ]:
train.groupby("era")["target"].mean().describe().round(4)

The average target is 0 in every era. The target is a rank **within** the era, so it only says how an asset did **compared with the others at that moment**. Your job is the same: put the assets of each era in order.

## 5. A first prediction

An idea to start from: an asset that jumped in the last hour tends to give a little of it back over the next few hours. Quants call this **short-term reversal**. So: predict the opposite of the last hour's return.

In [ ]:
def era_score(df, prediction="prediction", target="target"):
    """The competition metric: the mean over eras of the Spearman correlation between prediction and target."""
    per_era = df.groupby("era")[[prediction, target]].apply(
        lambda g: g[prediction].corr(g[target], method="spearman"))
    return per_era.fillna(0).mean()


recent = train[train["era"] > train["era"].max() - 1080].copy()  # roughly the last six months of train

rules = {
    "-ret_00 (reversal)": -recent["ret_00"],
    "+ret_00 (the opposite)": recent["ret_00"],
    "-ret_01 (one hour older)": -recent["ret_01"],
    "random guess": pd.Series(np.random.default_rng(0).random(len(recent)), index=recent.index),
}
pd.Series({name: round(era_score(recent.assign(prediction=p)), 4) for name, p in rules.items()}, name="score")

How to read these:

- A perfect order scores 1, a random guess about 0. In markets, a steady score clearly above 0 is a real signal.
- Flipping the sign flips the score.
- We check on the **most recent** training eras, because the test set comes after the training set in time.

You only get 5 submissions a day per team, so check ideas here first: the training data has the answers.

## 6. Make a submission

A submission is a CSV file with two columns, `id` and `prediction`, and one row for every row of `test`.

In [ ]:
submission = pd.DataFrame({"id": test["id"], "prediction": -test["ret_00"]})

# Check it before uploading: Kaggle rejects missing ids, duplicate ids, and non-numbers.
assert len(submission) == len(test), "one row per test row"
assert submission["id"].is_unique, "each id once"
assert submission["prediction"].notna().all(), "no empty predictions"

submission.to_csv("submission.csv", index=False)
submission.head()

To submit on Kaggle, save a version of your notebook and submit its output `submission.csv` to the competition, or download the file and upload it on the competition's **Submit Predictions** page.

Kaggle shows your **public** score right away. It covers only part of the test set. The **private** score decides the final ranking and appears after the deadline.

## 7. Your turn

1. Score a few other single columns on `recent`, with and without a minus sign. Which ones beat a random guess?
2. Pick your favorite, write it to `submission.csv`, and submit it.
3. Bonus: use `column_names` from section 1 to loop over a whole channel.

In [ ]:
recent["prediction"] = -recent["ret_00"]  # change this line
print("my rule on recent eras:", round(era_score(recent), 4))

## Recap

- A notebook runs Python cell by cell. pandas loads the data as a DataFrame.
- `df["col"]` picks a column, `df[condition]` picks rows, `groupby("era")` works era by era.
- A prediction is one number per row; only its **order within each era** is scored.
- Check locally on recent eras, then submit a checked `submission.csv`.

**Next week, Workshop 2:** exploring the data and understanding the metric. Questions are welcome on Discord. Dig well.